<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Answer:

I am continuing with Lane 4: CTR / Engagement Opportunity Scoring. I would frame this mainly as a ranking/scoring task. The system would give each page an opportunity score and rank the pages so that an SEO or content editor can review the most promising opportunities first.

Ranking is more useful here than simply classifying pages as good or bad because the main decision is which pages should be reviewed first.


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Answer:

The starter dataset does not contain a perfect observed label saying that a page is a real CTR opportunity, so for now I will use a proxy.

My provisional proxy will be the gap between a page's actual CTR and the typical CTR of pages in the same position tier. A larger positive gap means the page is receiving fewer clicks than other pages at similar ranking positions.

This is only a proxy and not ground truth. Later, with time-based data, I would prefer to validate the system using an observed outcome from a later time period rather than treating this proxy as the true answer.

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/jjcodes999/flyrank-ml-repo/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

# Keep pages with enough impressions and valid ranking-position data
lane = df[
    (df["impressions_90d"] >= 100) &
    (df["avg_position"] > 0)
].copy()

# Typical CTR for each position tier
lane["expected_ctr_tier"] = (
    lane.groupby("position_tier")["ctr"]
    .transform("median")
)

# Positive value = CTR is below the typical CTR for that position tier
lane["ctr_gap_proxy"] = (
    lane["expected_ctr_tier"] - lane["ctr"]
)

display(
    lane[
        ["position_tier", "ctr", "expected_ctr_tier", "ctr_gap_proxy"]
    ].head(10)
)

,position_tier,ctr,expected_ctr_tier,ctr_gap_proxy
0,striking,0.76,0.15,-0.61
1,page_3_5,0.05,0.06,0.01
2,page_3_5,0.09,0.06,-0.03
3,page_1,0.49,0.23,-0.26
4,page_3_5,0.13,0.06,-0.07
5,page_1,0.03,0.23,0.20
7,page_3_5,0.06,0.06,0.00
8,page_3_5,0.09,0.06,-0.03
9,page_1,0.16,0.23,0.07
10,top_3,1.55,0.19,-1.36


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Answer:

My main success metric would be Precision@20. This measures how many of the top 20 pages recommended for review are actually useful opportunities when checked against a later observed outcome or review.

As a provisional goal, I would consider a Precision@20 of 0.70 useful. This would mean that at least 14 of the 20 highest-ranked recommendations were useful. I would also expect the method to perform better than a simple fixed-rule baseline before claiming that the extra complexity of ML is worthwhile.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Answer:

My unit of analysis is one content page. Each row represents one page and contains information about its search visibility, clicks, CTR, ranking position, content properties and engagement.

For this lane, I am currently keeping pages with at least 100 impressions and valid ranking-position data so that very low-volume pages do not dominate the analysis. content_id identifies the page but would not be used as an ML feature

In [6]:
columns_to_show = [
    "content_id",
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "position_tier",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
    "expected_ctr_tier",
    "ctr_gap_proxy"
]

display(lane[columns_to_show].head(10))

print("Rows:", len(lane))
print("Unique pages:", lane["content_id"].nunique())
print("One row = one content page")

,content_id,content_type,impressions_90d,clicks_90d,ctr,avg_position,position_tier,content_age_days,days_since_last_update,engagement_rate,expected_ctr_tier,ctr_gap_proxy
0,content_304f48230142,keyword article,3803,29,0.76,10.6,striking,187,20,5.88,0.15,-0.61
1,content_a1fb4e703a9e,keyword article,15320,7,0.05,20.3,page_3_5,445,25,0.00,0.06,0.01
2,content_9aa793d4d895,keyword article,12581,11,0.09,36.5,page_3_5,141,20,0.00,0.06,-0.03
3,content_331d6c4de07b,keyword article,11751,58,0.49,6.2,page_1,463,22,1.28,0.23,-0.26
4,content_d99b7a2d90ca,keyword article,19140,24,0.13,44.0,page_3_5,263,14,0.00,0.06,-0.07
5,content_d4084a4bc775,keyword article,3970,1,0.03,8.5,page_1,147,20,0.00,0.23,0.20
7,content_a63219c6e95a,keyword article,1724,1,0.06,21.2,page_3_5,445,22,3.57,0.06,0.00
8,content_5e6c160719bc,keyword article,32574,29,0.09,46.0,page_3_5,90,20,5.88,0.06,-0.03
9,content_c27558df2b0c,keyword article,1240,2,0.16,4.9,page_1,257,104,0.00,0.23,0.07
10,content_d8ee6cc6d642,keyword article,20919,324,1.55,2.2,top_3,329,104,6.75,0.19,-1.36


Rows: 22006
Unique pages: 22006
One row = one content page


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Answer:

A fixed rule could flag pages using thresholds such as low CTR, high impressions and a strong ranking position. This would be a useful baseline, but it may be too simple because CTR opportunity can depend on several signals at the same time, including ranking position, impressions, content type, intent, age, freshness and engagement.

A ranking or scoring method could combine these signals instead of relying on one set of hard cutoffs. However, I would only say that ML is better if it performs better than the fixed-rule baseline on data it was not trained on. If it does not, the simpler rule would be preferable.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.